In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

%pip install neuralforecast
from neuralforecast import NeuralForecast
from neuralforecast.models import LSTM

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.0/294.0 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.2/348.2 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 831.6/831.6 kB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 MB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 63.2 MB/s eta 0:00:00


In [2]:
import pandas as pd

from google.colab import files
uploaded = files.upload()

Saving Exchange Rate_2017_2026.xlsx to Exchange Rate_2017_2026.xlsx


In [3]:
data = pd.read_excel('Exchange Rate_2017_2026.xlsx',sheet_name='Sheet2', header = 0)
data = data[['ds', 'y', 'unique_id']]
data.head()

,ds,y,unique_id
0,2017-01-02,13512.0,IDR=X
1,2017-01-03,13506.0,IDR=X
2,2017-01-04,13443.0,IDR=X
3,2017-01-05,13348.0,IDR=X
4,2017-01-06,13319.0,IDR=X


In [ ]:
data['ds'] = pd.to_datetime(data['ds'])

train_df = data[data['ds'] < '2025-07-04']
test_df  = data[data['ds'] >= '2025-07-04']

In [ ]:
import numpy as np

def objective(trial):

    # =============================
    # Hyperparameter
    # =============================

    learning_rate = trial.suggest_categorical(
        "learning_rate", [0.001, 0.005, 0.0001, 0.0005]
    )

    max_steps = trial.suggest_categorical(
        "max_steps", [100, 150]
    )

    input_size = trial.suggest_categorical(
        "input_size", [42, 84]
    )

    encoder_n_layers = trial.suggest_categorical(
        "encoder_n_layers", [4, 6]
    )

    decoder_layers = trial.suggest_categorical(
        "decoder_layers", [4, 6]
    )

    encoder_hidden_size = trial.suggest_categorical(
        "encoder_hidden_size", [400,500]
    )

    decoder_hidden_size = trial.suggest_categorical(
        "decoder_hidden_size", [400,500]
    )


    # =========================================
    # Time Series k-fold Cross Validation (K=5)
    # =========================================

    val_size = 151
    n_splits = 5

    initial_train_size = len(train_df) - (n_splits * val_size)

    mape_list = []

    for fold in range(n_splits):

        train_end = initial_train_size + fold * val_size

        train_fold = train_df.iloc[:train_end].copy()
        test_fold = train_df.iloc[
            train_end:train_end + val_size
        ].copy()

        horizon_fold = len(test_fold)

        # =============================
        # Model LSTM
        # =============================

        model = LSTM(
            h=horizon_fold,
            input_size=input_size,
            max_steps=max_steps,
            learning_rate=learning_rate,
            encoder_n_layers=encoder_n_layers,
            decoder_layers=decoder_layers,
            encoder_hidden_size=encoder_hidden_size,
            decoder_hidden_size=decoder_hidden_size,
            scaler_type="minmax",
        )

        nf = NeuralForecast(models=[model], freq="B")

        # =============================
        # Training Model LSTM
        # =============================

        nf.fit(df=train_fold)

        # =============================
        # Peramalan Data Uji
        # =============================

        fcst = nf.predict(df=train_fold)

        # =============================
        # Nilai Aktual & Peramalan
        # =============================

        y_pred = fcst["LSTM"].values
        y_true = test_fold["y"].values

        # =============================
        # Hitung MAPE
        # =============================

        mape_fold = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
        mape_list.append(mape_fold)

    # =============================
    # MAPE rata-rata semua fold
    # =============================

    return np.mean(mape_list)


In [ ]:
import optuna

study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42)
)

study.optimize(
    objective,
    n_trials=50
)

[I 2026-06-16 12:11:03,879] A new study created in memory with name: no-name-d0cc8c6c-b745-4925-b7a3-4613263296f0
INFO:lightning_fabric.utilities.seed:Seed set to 1
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name              | Type          | Params | Mode 
------------------------------------------------------------
0 | loss              | MAE           | 0      | train
1 | padder_train      | ConstantPad1d | 0      | train
2 | scaler            | TemporalNorm  | 0      | train
3 | hist_encoder      | LSTM          | 7.1 M  | train
4 | mlp_decoder       | MLP           | 1.2 M  | train
5 | upsample_sequence | Linear        | 12.8 K | train
------------------------------------------------------------
8.3 M     Trainable params
0   

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=100` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


... (1,741 outputs left out)

Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=100` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting: |          | 0/? [00:00<?, ?it/s]

[I 2026-06-16 13:34:29,660] Trial 49 finished with value: 2.325916743005801 and parameters: {'learning_rate': 0.0001, 'max_steps': 100, 'input_size': 42, 'encoder_n_layers': 4, 'decoder_layers': 4, 'encoder_hidden_size': 400, 'decoder_hidden_size': 400}. Best is trial 14 with value: 2.307705963151036.


In [ ]:
print("Best MAPE:", study.best_value)

print("Best Parameters:")
for k, v in study.best_params.items():
    print(f"{k}: {v}")

Best MAPE: 2.307705963151036
Best Parameters:
learning_rate: 0.0005
max_steps: 100
input_size: 42
encoder_n_layers: 4
decoder_layers: 4
encoder_hidden_size: 500
decoder_hidden_size: 500
